# Evaluación RAG — Fase 5: Validación Temporal y Robustez

Backtesting de las señales del RAG (Long/Short) y chequeo de robustez temporal. Reutiliza el **mismo archivo de predicciones de las Fases 2 y 3** (`id` + `sentimiento_predicho`) y `financial_ground_truth.csv` (Fase 3).

Métricas implementadas:
- **Backtesting Long-Short**: Bullish=Comprar, Bearish=Vender en corto, Sideways=no operar; P&L a `Price_t1/t5/t20`
- **Sharpe Ratio, Sortino Ratio, Maximum Drawdown** (global y por separado Long vs. Short)
- **Win Rate, Profit Factor**
- **Benchmark Buy & Hold** (comprar siempre, sin importar la señal)
- **Test de significancia** por bootstrap (¿el Sharpe real supera al de señales aleatorias?)
- **Costes de transacción** (opcional)
- **Desglose cronológico por ventana** — usando `walk_forward_windows.csv`, para inspeccionar si el rendimiento se degrada con el tiempo

### Documentos necesarios
1. `financial_ground_truth.csv` (de la Fase 3)
2. **El mismo archivo de predicciones de las Fases 2/3** (`id` + `sentimiento_predicho`)
3. `walk_forward_windows.csv` (adjunto) — 77 ventanas ya calculadas sobre las fechas reales de tu corpus, con tamaño de test adaptativo (≈18 noticias mínimo por ventana, en vez de un calendario fijo, porque 39 de los 157 meses del corpus tienen ≤2 noticias)

### ⚠️ Importante — qué SÍ y qué NO valida este notebook

Este notebook hace el **backtesting y las métricas de robustez** sobre las predicciones que le des. Pero la disciplina de **walk-forward real** (que el índice de Pinecone esté congelado a cada fecha de corte al generar cada predicción) depende de **cómo generaste tú el archivo de predicciones**:

- Si generaste las predicciones re-ejecutando tu RAG con el índice restringido a cada ventana de `walk_forward_windows.csv` → esto es un walk-forward real, y el desglose cronológico de la Sección 9 lo confirma.
- Si generaste las predicciones en una sola pasada con el índice completo actual → el desglose cronológico sigue siendo útil como inspección de estabilidad temporal, pero **no** es un walk-forward estricto (puede haber fuga de información de noticias futuras en el propio índice). Documenta cuál es tu caso en el informe final.

In [ ]:
# @title 1. Instalación de dependencias
!pip install -q pandas numpy scipy matplotlib

## 2. Subir archivos

In [ ]:
from google.colab import files
uploaded = files.upload()
print("Archivos subidos:", list(uploaded.keys()))

## 3. Configuración

In [ ]:
# @title Configuración
GROUND_TRUTH_PATH = "financial_ground_truth.csv"  # @param {type:"string"}
PREDICTIONS_PATH = "predicciones_sentimiento_sonnet.json"  # @param {type:"string"}
PREDICTIONS_FORMAT = "json"  # @param ["csv", "json"]
WINDOWS_PATH = "walk_forward_windows.csv"  # @param {type:"string"}

TRANSACTION_COST_PCT = 0.0005  # @param {type:"number"}  # 0.05% por operación; pon 0 para desactivar
N_BOOTSTRAP = 1000  # @param {type:"integer"}
HORIZONTES = {"1d": "price_t1", "5d": "price_t5", "20d": "price_t20"}

assert PREDICTIONS_PATH, "Indica el nombre del archivo de predicciones que subiste."


## 4. Cargar, unir y filtrar

In [ ]:
import re
import pandas as pd
import numpy as np

LABEL_MAP = {
    "bearish": "bearish", "bajista": "bearish", "bajo": "bearish", "negativo": "bearish",
    "bear": "bearish", "down": "bearish", "venta": "bearish", "vender": "bearish",
    "bullish": "bullish", "alcista": "bullish", "alto": "bullish", "positivo": "bullish",
    "bull": "bullish", "up": "bullish", "compra": "bullish", "comprar": "bullish",
    "sideways": "sideways", "lateral": "sideways", "neutral": "sideways", "neutro": "sideways",
    "estable": "sideways", "flat": "sideways", "sin cambios": "sideways",
}

def normalize_label(raw):
    if raw is None:
        return None
    key = str(raw).strip().lower()
    key = re.sub(r"[^a-záéíóúñ ]", "", key)
    return LABEL_MAP.get(key, None)

gt = pd.read_csv(GROUND_TRUTH_PATH, parse_dates=["fecha"])
if PREDICTIONS_FORMAT == "csv":
    preds = pd.read_csv(PREDICTIONS_PATH)
else:
    preds = pd.read_json(PREDICTIONS_PATH)

preds["sentimiento_predicho_norm"] = preds["sentimiento_predicho"].apply(normalize_label)

df = gt.merge(preds[["id", "sentimiento_predicho_norm"]], on="id", how="inner")
df = df.dropna(subset=["sentimiento_predicho_norm"])
n_antes = len(df)
df = df[df["enrich_status"] == "ok"].copy()
df = df.sort_values("fecha").reset_index(drop=True)

print(f"Filas tras unión con predicciones: {n_antes}")
print(f"Filas tras excluir enrich_status != 'ok': {len(df)}")
print(f"Señales: {df['sentimiento_predicho_norm'].value_counts().to_dict()}")

## 5. Mecánica de trading y cálculo de P&L por horizonte

Bullish → Long, Bearish → Short, Sideways → sin operación. Cierre en `Price_t1` (intradía), `Price_t5` (swing) o `Price_t20` (posicional).

In [ ]:
def compute_pnl(row, price_col):
    p0, ph = row["price_t0"], row[price_col]
    if pd.isna(p0) or pd.isna(ph) or p0 == 0:
        return np.nan
    raw_return = (ph - p0) / p0
    if row["sentimiento_predicho_norm"] == "bullish":
        pnl = raw_return
    elif row["sentimiento_predicho_norm"] == "bearish":
        pnl = -raw_return
    else:
        return np.nan  # sideways: no operar
    if TRANSACTION_COST_PCT:
        pnl -= TRANSACTION_COST_PCT
    return pnl

for h, price_col in HORIZONTES.items():
    df[f"pnl_{h}"] = df.apply(lambda r: compute_pnl(r, price_col), axis=1)

df[["id", "ticker", "fecha", "sentimiento_predicho_norm", "pnl_1d", "pnl_5d", "pnl_20d"]].head(10)

## 6. Métricas de la estrategia por horizonte

Sharpe y Sortino se anualizan aproximando la frecuencia de operación por horizonte (252 días bursátiles / duración de la posición). Maximum Drawdown se calcula sobre una curva de equity simulada, asumiendo capital compuesto secuencialmente en el orden cronológico de las señales (simplificación estándar de backtesting simple, ya que las operaciones no necesariamente son secuenciales/no solapadas en la realidad).

In [ ]:
PERIODS_PER_YEAR = {"1d": 252, "5d": 252 / 5, "20d": 252 / 20}

def sharpe_ratio(returns, periods_per_year):
    returns = returns.dropna()
    if len(returns) < 2 or returns.std() == 0:
        return np.nan
    return (returns.mean() / returns.std()) * np.sqrt(periods_per_year)

def sortino_ratio(returns, periods_per_year):
    returns = returns.dropna()
    downside = returns[returns < 0]
    if len(returns) < 2 or len(downside) == 0 or downside.std() == 0:
        return np.nan
    return (returns.mean() / downside.std()) * np.sqrt(periods_per_year)

def max_drawdown(returns):
    returns = returns.dropna()
    if len(returns) == 0:
        return np.nan
    equity = (1 + returns).cumprod()
    running_max = equity.cummax()
    drawdown = (equity - running_max) / running_max
    return drawdown.min()

def win_rate(returns):
    returns = returns.dropna()
    if len(returns) == 0:
        return np.nan
    return (returns > 0).mean()

def profit_factor(returns):
    returns = returns.dropna()
    gains = returns[returns > 0].sum()
    losses = -returns[returns < 0].sum()
    if losses == 0:
        return np.nan
    return gains / losses

strategy_rows = []
for h in HORIZONTES:
    col = f"pnl_{h}"
    all_returns = df[col]
    long_returns = df.loc[df["sentimiento_predicho_norm"] == "bullish", col]
    short_returns = df.loc[df["sentimiento_predicho_norm"] == "bearish", col]

    strategy_rows.append({
        "horizonte": h,
        "n_operaciones": all_returns.notna().sum(),
        "sharpe": sharpe_ratio(all_returns, PERIODS_PER_YEAR[h]),
        "sortino": sortino_ratio(all_returns, PERIODS_PER_YEAR[h]),
        "max_drawdown": max_drawdown(all_returns),
        "max_drawdown_long": max_drawdown(long_returns),
        "max_drawdown_short": max_drawdown(short_returns),
        "win_rate": win_rate(all_returns),
        "profit_factor": profit_factor(all_returns),
    })

strategy_df = pd.DataFrame(strategy_rows)
strategy_df

## 7. Benchmark Buy & Hold

Retorno medio por operación si siempre se hubiera comprado (long), sin importar la señal del RAG — la referencia mínima que el sistema debe superar.

In [ ]:
benchmark_rows = []
for h, price_col in HORIZONTES.items():
    raw_returns = (df[price_col] - df["price_t0"]) / df["price_t0"]
    benchmark_rows.append({
        "horizonte": h,
        "buy_and_hold_mean_return": raw_returns.mean(),
        "buy_and_hold_sharpe": sharpe_ratio(raw_returns, PERIODS_PER_YEAR[h]),
    })

benchmark_df = pd.DataFrame(benchmark_rows)
comparativa = strategy_df[["horizonte", "sharpe"]].merge(benchmark_df, on="horizonte")
comparativa["alpha_sharpe"] = comparativa["sharpe"] - comparativa["buy_and_hold_sharpe"]
comparativa

## 8. Test de significancia estadística (bootstrap)

Compara el Sharpe real contra el que generarían señales aleatorias con el mismo timing y proporción de Long/Short/sin-operar. Si el Sharpe real cae dentro del rango que produciría el azar, el sistema no está demostrando valor añadido real.

In [ ]:
def bootstrap_pvalue(df, horizonte, price_col, n_iter=N_BOOTSTRAP, seed=42):
    rng = np.random.default_rng(seed)
    real_returns = df[f"pnl_{horizonte}"]
    real_sharpe = sharpe_ratio(real_returns, PERIODS_PER_YEAR[horizonte])

    original_labels = df["sentimiento_predicho_norm"].values.copy()
    random_sharpes = []

    for _ in range(n_iter):
        shuffled = rng.permutation(original_labels)
        raw_return = (df[price_col] - df["price_t0"]) / df["price_t0"]
        pnl = np.where(shuffled == "bullish", raw_return,
              np.where(shuffled == "bearish", -raw_return, np.nan))
        if TRANSACTION_COST_PCT:
            pnl = np.where(~np.isnan(pnl), pnl - TRANSACTION_COST_PCT, pnl)
        random_sharpes.append(sharpe_ratio(pd.Series(pnl), PERIODS_PER_YEAR[horizonte]))

    random_sharpes = np.array([s for s in random_sharpes if not np.isnan(s)])
    p_value = (random_sharpes >= real_sharpe).mean()
    return real_sharpe, random_sharpes, p_value


significance_rows = []
bootstrap_distributions = {}
for h, price_col in HORIZONTES.items():
    real_sharpe, random_sharpes, p_value = bootstrap_pvalue(df, h, price_col)
    bootstrap_distributions[h] = random_sharpes
    significance_rows.append({
        "horizonte": h, "sharpe_real": real_sharpe,
        "sharpe_aleatorio_medio": random_sharpes.mean(), "sharpe_aleatorio_p95": np.percentile(random_sharpes, 95),
        "p_valor": p_value, "significativo (p<0.05)": p_value < 0.05,
    })

significance_df = pd.DataFrame(significance_rows)
significance_df

## 9. Desglose cronológico por ventana de Walk-Forward

Vigila si el rendimiento se degrada con el tiempo (una señal de sobreajuste al periodo histórico concreto, o de que el modelo de embeddings/LLM ya "conocía" ciertos eventos de su propio pre-entrenamiento).

In [ ]:
windows = pd.read_csv(WINDOWS_PATH, parse_dates=["test_desde", "test_hasta"])

def asignar_ventana(fecha):
    match = windows[(windows["test_desde"] <= fecha) & (fecha <= windows["test_hasta"])]
    return match["ventana"].iloc[0] if len(match) else np.nan

df["ventana"] = df["fecha"].apply(asignar_ventana)

ventana_rows = []
for ventana, sub in df.groupby("ventana"):
    row = {"ventana": ventana, "fecha_inicio": sub["fecha"].min().date(), "n_señales": len(sub)}
    for h in HORIZONTES:
        row[f"sharpe_{h}"] = sharpe_ratio(sub[f"pnl_{h}"], PERIODS_PER_YEAR[h])
        row[f"win_rate_{h}"] = win_rate(sub[f"pnl_{h}"])
    ventana_rows.append(row)

ventana_df = pd.DataFrame(ventana_rows).sort_values("ventana")
ventana_df

## 10. Visualización — Equity curve y evolución temporal

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 1, figsize=(11, 9))

# Equity curve (horizonte 5d como referencia de swing trading)
equity = (1 + df["pnl_5d"].fillna(0)).cumprod()
axes[0].plot(df["fecha"], equity, color="#2E5395")
axes[0].set_title("Curva de equity simulada — estrategia swing (5 días)")
axes[0].set_ylabel("Capital (base=1)")

# Sharpe por ventana a lo largo del tiempo
axes[1].plot(ventana_df["ventana"], ventana_df["sharpe_5d"], marker="o", color="#B5474D")
axes[1].axhline(0, color="black", linewidth=0.8)
axes[1].set_title("Sharpe Ratio (5d) por ventana de Walk-Forward (orden cronológico)")
axes[1].set_xlabel("Ventana (cronológico)")
axes[1].set_ylabel("Sharpe (5d)")

plt.tight_layout()
plt.savefig("robustez_temporal.png", dpi=150)
plt.show()

## 11. Guardar resultados

In [ ]:
with pd.ExcelWriter("resultados_temporal_robustez.xlsx") as writer:
    strategy_df.to_excel(writer, sheet_name="metricas_estrategia", index=False)
    comparativa.to_excel(writer, sheet_name="vs_buy_and_hold", index=False)
    significance_df.to_excel(writer, sheet_name="test_significancia", index=False)
    ventana_df.to_excel(writer, sheet_name="desglose_por_ventana", index=False)

from google.colab import files as colab_files
colab_files.download("resultados_temporal_robustez.xlsx")
colab_files.download("robustez_temporal.png")

print("Guardado resultados_temporal_robustez.xlsx con 4 hojas.")